In [ ]:
# pip install roboflow ultralytics supervision jupyter ipykernel

In [ ]:
from roboflow import download_dataset
from ultralytics import YOLO
import supervision as sv


WORKSPACE = "tutorial-extpj"              
PROJECT = "brain-tumor-segmentation-jteuo"   
VERSION = 1                           

dataset_url = f"https://universe.roboflow.com/{WORKSPACE}/{PROJECT}/dataset/{VERSION}"

In [ ]:
dataset = download_dataset(dataset_url, 'yolov8', location="../data/Brain-Tumor-Segmentation")

def train_segmentation_model():
    model = YOLO("../yolov8s-seg.pt")

    results = model.train(
        data=f'{dataset.location}/data.yaml', 
        epochs=150,
        patience=20,
        project='runs/segment',
        name='train',
        exist_ok=True,
        verbose=False
    )
    return results, model

In [ ]:
results, model = train_segmentation_model()

In [ ]:
dataset = download_dataset(dataset_url, 'yolov8', location="../data/Brain-Tumor-Segmentation")
model = YOLO('../runs/segment/train5/weights/best.pt')

In [ ]:
import random
import supervision as sv

ds = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{dataset.location}/valid/images",
    annotations_directory_path=f"{dataset.location}/valid/labels",
    data_yaml_path=f"{dataset.location}/data.yaml"
)

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Get a random image from the dataset
i = random.randint(0, len(ds) - 1)
image_path, image, target = ds[i]

# Perform a prediction on the selected image
results = model(image)  # Pass the image to the YOLO model

# Results is a list, so we need to get the first item
result = results[0]

# Accessing the segmentation masks
masks = result.masks  # Get the Masks object containing the detected segmentation masks

# Access the raw mask data (this is likely a tensor or numpy array)
masks_data = masks.data.numpy()  # Using .data to extract the raw tensor data and converting it to numpy

# Ensure the image is in the correct format for displaying (height, width, channels)
# If the image is in (C, H, W) format, you can convert it to (H, W, C) format for plotting
if image.ndim == 3 and image.shape[0] == 3:  # Assuming (C, H, W) format for image
    image = np.transpose(image, (1, 2, 0))  # Convert to (H, W, C)

# Plot the original image and the image with the mask overlay
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Plot original image
axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis('off')  # Hide axes

# Plot image with segmentation mask overlay
axes[1].imshow(image)  # Display the original image
axes[1].imshow(masks_data[0], cmap='jet', alpha=0.5)  # Overlay the first mask with some transparency
axes[1].set_title("Image with Segmentation Mask")
axes[1].axis('off')  # Hide axes

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline

# Extract bounding box coordinates
x1, y1, x2, y2 = result.boxes.xyxy[0].tolist()  # Get the coordinates of the bounding box

# Plot the image
plt.imshow(image)
ax = plt.gca()

# Create a Rectangle patch for the bounding box
rect = patches.Rectangle(
    (x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='r', facecolor='none'
)

# Add the rectangle to the plot
ax.add_patch(rect)

# Title and show the plot
plt.title("Bounding Box on Image")
plt.show()




In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Get a random image from the dataset
i = random.randint(0, len(ds) - 1)
image_path, image, target = ds[i]

# Perform a prediction on the selected image
results = model(image)  # Pass the image to the YOLO model

# Results is a list, so we need to get the first item
result = results[0]

# Accessing the segmentation masks
masks = result.masks  # Get the Masks object containing the detected segmentation masks

# Access the raw mask data (this is likely a tensor or numpy array)
masks_data = masks.data.numpy()  # Using .data to extract the raw tensor data and converting it to numpy

# Ensure the image is in the correct format for displaying (height, width, channels)
# If the image is in (C, H, W) format, you can convert it to (H, W, C) format for plotting
if image.ndim == 3 and image.shape[0] == 3:  # Assuming (C, H, W) format for image
    image = np.transpose(image, (1, 2, 0))  # Convert to (H, W, C)

# Get the predicted label based on the highest confidence score
# Access predicted class from result.boxes.cls
predicted_class_idx = int(result.boxes.cls[0].item())  # Get the class index
predicted_class = result.names[predicted_class_idx]  # Map index to class name

# Plot the original image and the image with the mask overlay
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Plot original image with the predicted label
axes[0].imshow(image)
axes[0].set_title(f"Original Image - {predicted_class}")
axes[0].axis('off')  # Hide axes

# Create a colored mask (you can change the color here)
colored_mask = np.zeros_like(image)  # Same size as the image
colored_mask[masks_data[0] > 0] = [255, 0, 0]  # Red color where the mask is

# Plot image with the segmentation mask overlay using transparency
axes[1].imshow(image)  # Display the original image
axes[1].imshow(colored_mask, alpha=0.5)  # Overlay the colored mask with transparency
axes[1].set_title(f"Prediction - {predicted_class}")
axes[1].axis('off')  # Hide axes

plt.tight_layout()
plt.show()
